# 📊 Tüm BIST (~606 Hisse) HMM+MOM3 — Düzeltilmiş Giriş Zamanlaması
## Veri: TradingView (tvdatafeed) | Evren: Tüm BIST | Timeframe: Haftalık

**Temel Düzeltme:** Zirvedeki hisseyi değil, **boğa trendinde geri çekilmiş** hisseyi sinyal ver.

| Eski Sorun | Yeni Çözüm |
|---|---|
| Tüm lagging göstergeler yükselen hisseyi işaretler | Overextension + RSI + 52H zirve cezaları eklendi |
| MOM yüksek = daha iyi skor | MOM ideal aralık: %10-60 (parabolic hareketler cezalı) |
| Zirvedeki hisse GÜÇLÜ AL verir | Zirveye yakın + RSI>70 → AŞIRI DEĞER uyarısı |
| Geç giriş riski | DİP FIRSATI sinyali: boğa rejiminde pullback = en iyi giriş |

**Sinyal Hiyerarşisi (En İyi → En Kötü):**  
`DİP FIRSATI` > `GÜÇLÜ AL` > `AL` > `HMM AL` > `MOM AL` > `BEKLE` > `AŞIRI DEĞER`


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg, imp in [
    ("git+https://github.com/rongardF/tvdatafeed.git", "tvDatafeed"),
    ("hmmlearn", "hmmlearn"),
    ("tqdm", "tqdm"),
    ("openpyxl", "openpyxl"),
]:
    try:
        __import__(imp)
    except ImportError:
        print(f"{imp} kuruluyor...")
        pip(pkg)

print("✅ Tüm kütüphaneler hazır.")


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import logging
logging.getLogger("hmmlearn").setLevel(logging.ERROR)  # WARNING logları consola basmasın
import numpy as np
import pandas as pd
import requests, time, os, pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.linalg import inv
from hmmlearn.hmm import GaussianHMM
from tvDatafeed import TvDatafeed, Interval
from tqdm.notebook import tqdm

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 170)
print("✅ Import'lar tamam.")


In [ ]:
INTERVAL   = Interval.in_weekly
N_BARS     = 200          # ~4 yıl haftalık veri
EXCHANGE   = "BIST"
BENCHMARK  = "XU100"

CACHE_DATA  = True
CACHE_FILE  = "bist_weekly_cache.pkl"

MIN_BARS    = 52
MAX_SYMBOLS = 700
TRAIN_MIN   = 52
TEST_SIZE   = 13
STEP        = 13

# TradingView hesabı (opsiyonel)
TV_USERNAME = ""
TV_PASSWORD = ""

# ── Giriş Zamanlaması Filtreleri ──────────────────────────────────────────
RSI_OVERBOUGHT   = 70    # RSI bu değerin üstündeyse AL verme → AŞIRI DEĞER
RSI_IDEAL_MAX    = 65    # İdeal alım bölgesi üst sınırı
OVEREXT_THRESH   = 0.20  # Fiyat SMA26'nın %20+ üzerindeyse penaltı başlar
TOP_PROXIMITY    = 0.05  # 52H zirvesine %5'ten yakınsa penaltı
MOM13_IDEAL_MIN  = 0.05  # 13H momentum ideal alt sınırı (%5)
MOM13_IDEAL_MAX  = 0.70  # 13H momentum ideal üst sınırı (%70) — üstü parabolic

# ── Kalite Filtreleri — Sinyal Azaltma ───────────────────────────────────
# Problem: boğa piyasasında 300+ hisse AL sinyali verebilir → anlamsız
# Çözüm  : sadece skor yeterince yüksek ve WFO geçmişi pozitif hisseler listede
MIN_COMPOSITE    = 52    # Bu skorun altındaki hisseler → sinyal BEKLE'ye düşer
MIN_WFO_SHARPE   = 0.10  # WFO Sharpe bu altındaysa tarihsel kanıt yetersiz
TOP_N_PER_SIG    = 15    # Her sinyal kategorisinde gösterilecek maksimum hisse

print("✅ Konfigürasyon yüklendi.")
print(f"   RSI overbought eşiği : {RSI_OVERBOUGHT}")
print(f"   Overextension eşiği  : %{OVEREXT_THRESH*100:.0f} üzeri SMA26")
print(f"   İdeal MOM13W aralığı : %{MOM13_IDEAL_MIN*100:.0f} – %{MOM13_IDEAL_MAX*100:.0f}")
print()
print(f"   KALİTE FİLTRESİ:")
print(f"   Min. Kompozit Skor   : {MIN_COMPOSITE}/100  (altı → BEKLE)")
print(f"   Min. WFO Sharpe      : {MIN_WFO_SHARPE}     (altı → güvenilir geçmiş yok)")
print(f"   Kategori başı maks.  : {TOP_N_PER_SIG} hisse")


In [ ]:
tv = TvDatafeed(TV_USERNAME, TV_PASSWORD) if TV_USERNAME else TvDatafeed()

try:
    _t = tv.get_hist("THYAO", EXCHANGE, interval=INTERVAL, n_bars=5)
    print(f"✅ TvDatafeed bağlantısı OK — THYAO {len(_t)} bar")
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")


In [ ]:
def get_all_bist():
    url = "https://scanner.tradingview.com/turkey/scan"
    payload = {
        "filter": [{"left":"type","operation":"equal","right":"stock"}],
        "options": {"lang":"tr"},
        "symbols": {"query":{"types":["stock"]},"tickers":[]},
        "columns": ["name","close","volume","market_cap_basic","average_volume_10d_calc"],
        "sort": {"sortBy":"market_cap_basic","sortOrder":"desc"},
        "range": [0, MAX_SYMBOLS],
    }
    try:
        r = requests.post(url, json=payload, timeout=30,
                          headers={"User-Agent":"Mozilla/5.0"})
        data = r.json().get("data", [])
        syms = []
        for row in data:
            s = row.get("s","")
            if s.startswith("BIST:"):
                syms.append(s.split(":")[1])
        print(f"✅ TradingView Scanner: {len(syms)} BIST sembolü")
        return syms
    except Exception as e:
        print(f"❌ Scanner hatası: {e} — yedek liste kullanılıyor")
        return None

ALL_SYMBOLS = get_all_bist()
if ALL_SYMBOLS is None:
    ALL_SYMBOLS = [
        "THYAO","GARAN","ASELS","BIMAS","EREGL","KCHOL","AKBNK","TUPRS",
        "FROTO","SISE","HALKB","VAKBN","MGROS","ASTOR","TKFEN","ISCTR",
        "TOASO","CCOLA","ENKAI","SAHOL","YKBNK","TCELL","PETKM","PGSUS",
        "KOZAL","OYAKC","ARCLK","KRDMD","DOHOL","SASA","TTKOM","AGHOL",
        "ULKER","AEFES","EKGYO","TAVHL","MAVI","BRSAN","GESAN","CWENE",
        "EUPWR","FENER","MIATK","PATEK","QUAGR","KTLEV","CVKMD","VESTL",
    ]

print(f"Toplam taranacak: {len(ALL_SYMBOLS)} hisse")


In [ ]:
def safe_get(sym, retries=3):
    for i in range(retries):
        try:
            df = tv.get_hist(sym, EXCHANGE, interval=INTERVAL, n_bars=N_BARS)
            if df is not None and len(df) >= MIN_BARS:
                df.columns = [c.lower() for c in df.columns]
                return df.sort_index()[~df.sort_index().index.duplicated()]
        except: time.sleep(1.5**i)
    return None

RAW = {}
if CACHE_DATA and os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f: RAW = pickle.load(f)
    print(f"📂 Önbellekten yüklendi: {len(RAW)} hisse")
    to_dl = [s for s in ALL_SYMBOLS if s not in RAW]
else:
    to_dl = ALL_SYMBOLS

print("\nXU100 indiriliyor...")
xu100_raw = safe_get(BENCHMARK)
XU100 = xu100_raw["close"].dropna() if xu100_raw is not None else None
print(f"✅ XU100: {len(XU100) if XU100 is not None else 0} bar")

if to_dl:
    failed = []
    print(f"\n{len(to_dl)} hisse indiriliyor...")
    for sym in tqdm(to_dl, desc="Download"):
        df = safe_get(sym)
        if df is not None: RAW[sym] = df
        else: failed.append(sym)
        time.sleep(0.25)
    if CACHE_DATA:
        with open(CACHE_FILE,"wb") as f: pickle.dump(RAW, f)
        print(f"💾 Cache kaydedildi: {CACHE_FILE}")
    print(f"✅ Başarılı: {len(RAW)} | Başarısız: {len(failed)}")


In [ ]:
def prepare(df_raw, xu100=None):
    df = df_raw[["open","high","low","close","volume"]].copy()
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df.dropna(subset=["close"], inplace=True)
    if len(df) < MIN_BARS: return None

    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    # Split tespiti
    df.loc[df["log_ret"] < -0.45, "log_ret"] = np.nan

    # Göreceli güç
    if xu100 is not None:
        xu = xu100.reindex(df.index, method="ffill")
        df["rel_xu100"] = df["log_ret"] - np.log(xu/xu.shift(1))
    else:
        df["rel_xu100"] = df["log_ret"] - df["log_ret"].rolling(26).mean()

    # Momentum
    df["mom3w"]  = df["close"].pct_change(3)
    df["mom13w"] = df["close"].pct_change(13)

    # Hareketli ortalamalar
    df["sma13"] = df["close"].rolling(13).mean()
    df["sma26"] = df["close"].rolling(26).mean()

    # 52 haftalık yüksek/düşük (overextension tespiti için)
    df["high52w"] = df["close"].rolling(52).max()
    df["low52w"]  = df["close"].rolling(52).min()

    # Fiyatın SMA26'dan uzaklığı
    df["ext_ratio"] = df["close"] / df["sma26"].clip(lower=1e-9) - 1

    # RSI(14)
    d = df["close"].diff()
    df["rsi14"] = 100 - 100/(1 + d.clip(lower=0).rolling(14).mean() /
                              (-d.clip(upper=0)).rolling(14).mean().clip(lower=1e-9))

    # ATR(14)
    tr = pd.concat([
        df["high"]-df["low"],
        (df["high"]-df["close"].shift(1)).abs(),
        (df["low"]-df["close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14).mean()

    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean().clip(lower=1)
    return df

WEEKLY = {}
for t, raw in RAW.items():
    try:
        df = prepare(raw, XU100)
        if df is not None: WEEKLY[t] = df
    except: pass

print(f"✅ {len(WEEKLY)} hisse için indikatörler hazır.")


In [ ]:
class HMMStrategy:
    def __init__(self, n_iter=200, min_samples=30):
        self.n_iter, self.min_samples = n_iter, min_samples
        self.model, self.bull_state = None, 0

    def _X(self, df):
        return df[["log_ret","rel_xu100"]].replace([np.inf,-np.inf],np.nan).dropna()

    def fit(self, df):
        X = self._X(df)
        if len(X) < self.min_samples: return False
        try:
            m = GaussianHMM(n_components=2, covariance_type="diag",
                            n_iter=self.n_iter, tol=1e-2,
                            random_state=42)
            m.fit(X.values)
            self.model = m
            self.bull_state = int(np.argmax(m.means_[:,0]))
            return True
        except: return False

    def predict_states(self, df):
        if not self.model: return pd.Series(0, index=df.index)
        X = self._X(df)
        if X.empty: return pd.Series(0, index=df.index)
        try:
            s = self.model.predict(X.values)
            return pd.Series((s==self.bull_state).astype(float),
                             index=X.index).reindex(df.index, fill_value=0)
        except: return pd.Series(0, index=df.index)

    def bull_prob(self, df):
        if not self.model: return 0.5
        X = self._X(df)
        if X.empty: return 0.5
        try:
            return float(self.model.predict_proba(X.values)[-1, self.bull_state])
        except: return 0.5

    def weeks_in_current_regime(self, df):
        """Son rejime kaç haftadır devam ediliyor?"""
        states = self.predict_states(df)
        cur = int(states.iloc[-1])
        count = 0
        for v in reversed(states.values):
            if int(v) == cur: count += 1
            else: break
        return count


def wfo_hmm(df):
    n = len(df)
    oos_r, oos_d = [], []
    for fs in range(TRAIN_MIN, n - TEST_SIZE + 1, STEP):
        h = HMMStrategy()
        if not h.fit(df.iloc[:fs]): continue
        sig = h.predict_states(df.iloc[:fs+TEST_SIZE])
        sig_t = sig.iloc[fs:fs+TEST_SIZE].shift(1).fillna(0)
        ret_t = df["log_ret"].iloc[fs:fs+TEST_SIZE].fillna(0)
        oos_r.extend((sig_t*ret_t).tolist())
        oos_d.extend(ret_t.index.tolist())
    if not oos_r: return {"sharpe":-99, "ret":-99}
    s = pd.Series(oos_r, index=oos_d).sort_index()
    return {"sharpe": round(float(s.mean()/(s.std()+1e-9))*np.sqrt(52),3),
            "ret":    round(float(np.exp(s.sum())-1)*100, 1)}


def kalman_proj(close_s, n_fw=13):
    y = np.log(close_s.dropna().values)
    if len(y) < 20:
        p = float(close_s.iloc[-1]); return 0, p, p, p
    F=np.array([[1,1],[0,1]]); H=np.array([[1,0]])
    Q=np.eye(2)*1e-4; R=np.array([[1e-2]])
    x=np.array([[y[0]],[0.]]); P=np.eye(2)
    for obs in y:
        xp=F@x; Pp=F@P@F.T+Q; inn=obs-(H@xp)[0,0]
        S=H@Pp@H.T+R; K=Pp@H.T@inv(S); x=xp+K*inn; P=(np.eye(2)-K@H)@Pp
    vel=float(x[1,0]); lev=float(x[0,0])
    fl=lev+vel*n_fw; sig=float(np.std(np.diff(y))*np.sqrt(n_fw))
    return (round(vel*52*100,2), round(float(np.exp(fl)),2),
            round(float(np.exp(fl-2*sig)),2), round(float(np.exp(fl+2*sig)),2))

print("✅ HMM | WFO | Kalman modelleri hazır.")


In [ ]:
def scan_ticker(ticker):
    df = WEEKLY.get(ticker)
    if df is None: return None
    df = df.dropna(subset=["log_ret","rel_xu100"])
    if len(df) < MIN_BARS: return None

    # ── Temel değerler ──────────────────────────────────────────────────────
    cur   = float(df["close"].iloc[-1])
    if cur <= 0: return None
    atr   = float(df["atr14"].iloc[-1])    if pd.notna(df["atr14"].iloc[-1])    else cur * 0.03
    rsi   = float(df["rsi14"].iloc[-1])    if pd.notna(df["rsi14"].iloc[-1])    else 50.0
    vol   = float(df["vol_ratio"].iloc[-1]) if pd.notna(df["vol_ratio"].iloc[-1]) else 1.0
    mom13 = float(df["mom13w"].iloc[-1])   if pd.notna(df["mom13w"].iloc[-1])   else 0.0
    mom3  = float(df["mom3w"].iloc[-1])    if pd.notna(df["mom3w"].iloc[-1])    else 0.0
    sma13 = float(df["sma13"].iloc[-1])    if pd.notna(df["sma13"].iloc[-1])    else cur
    sma26 = float(df["sma26"].iloc[-1])    if pd.notna(df["sma26"].iloc[-1])    else cur
    ext   = float(df["ext_ratio"].iloc[-1]) if pd.notna(df["ext_ratio"].iloc[-1]) else 0.0
    h52   = float(df["high52w"].iloc[-1])  if pd.notna(df["high52w"].iloc[-1])  else cur
    h52   = max(h52, cur)   # floating-point guard

    atr_pct = round(atr / cur * 100, 1)

    wfo = wfo_hmm(df)

    # ── HMM ─────────────────────────────────────────────────────────────────
    hmm = HMMStrategy()
    hmm.fit(df)
    prob          = hmm.bull_prob(df)
    hmm_sig       = prob > 0.55
    weeks_in_bull = hmm.weeks_in_current_regime(df) if hmm_sig else 0

    # ── Kalman ──────────────────────────────────────────────────────────────
    vel_pct, tgt, tgt_lo, tgt_hi = kalman_proj(df["close"])
    tgt    = tgt if tgt > 0 else cur
    upside = (tgt / cur - 1) * 100

    # ── Giriş kalitesi metrikleri ────────────────────────────────────────────
    top_dist_pct = max(0.0, (h52 - cur) / h52 * 100)
    ext_pct      = ext * 100
    is_pullback  = (mom3 < -0.02) and (mom13 > MOM13_IDEAL_MIN)
    is_fresh     = 0 < weeks_in_bull <= 4
    is_parabolic = mom13 > MOM13_IDEAL_MAX

    # ── Tükenme Tespiti ──────────────────────────────────────────────────────
    mom_slowing   = (mom3 < mom13 * 0.30) and (mom13 > 0)
    vol_declining = (vol < 0.75)

    cond_a = (rsi > RSI_OVERBOUGHT) and (mom_slowing or vol_declining)
    cond_b = is_parabolic            and (mom_slowing or vol_declining)
    cond_c = (ext_pct > 35)          and (rsi > 65)
    is_overvalued = cond_a or cond_b or cond_c

    # ── Kompozit Skor (0-100) ────────────────────────────────────────────────
    s_hmm = prob * 40

    if MOM13_IDEAL_MIN <= mom13 <= MOM13_IDEAL_MAX:
        s_mom = 20.0
    elif mom13 < MOM13_IDEAL_MIN:
        s_mom = max(0.0, mom13 / MOM13_IDEAL_MIN * 20)
    else:
        excess = (mom13 - MOM13_IDEAL_MAX) / MOM13_IDEAL_MAX
        s_mom  = max(0.0, 20.0 - excess * 40)

    s_kal = min(max(vel_pct / 40.0 * 15, 0.0), 15.0)

    s_entry  = 15.0
    s_entry -= min(max(ext_pct - 10, 0) * 0.5, 10.0)
    s_entry -= min(max(rsi - 55, 0) * 0.3, 8.0)
    s_entry -= max(0.0, TOP_PROXIMITY * 100 - top_dist_pct)
    s_entry -= 5.0 if mom_slowing  else 0.0
    s_entry -= 3.0 if vol_declining else 0.0
    s_entry  = max(0.0, s_entry)

    s_ma  = 5.0 if sma13 > sma26 else 0.0
    s_vol = min(vol * 2.5, 5.0)
    bonus  = 5.0 if is_pullback else 0.0
    bonus += 3.0 if is_fresh    else 0.0

    composite = round(max(0.0, min(100.0,
        s_hmm + s_mom + s_kal + s_entry + s_ma + s_vol + bonus)), 1)

    # ── Sinyal Mantığı + Kalite Filtresi ────────────────────────────────────
    # Adım 1: Teknik koşullara göre temel sinyal
    if   is_overvalued and hmm_sig:
        sinyal = "AŞIRI DEĞER"
    elif hmm_sig and is_pullback and rsi < RSI_OVERBOUGHT:
        sinyal = "DİP FIRSATI"
    elif (hmm_sig and MOM13_IDEAL_MIN < mom13 <= MOM13_IDEAL_MAX
          and rsi < RSI_OVERBOUGHT and ext_pct < 20 and sma13 > sma26):
        sinyal = "GÜÇLÜ AL"
    elif hmm_sig and mom13 > 0 and rsi < RSI_OVERBOUGHT and ext_pct < OVEREXT_THRESH * 100:
        sinyal = "AL"
    elif hmm_sig and rsi < RSI_OVERBOUGHT:
        sinyal = "HMM AL"
    elif mom13 > MOM13_IDEAL_MIN and sma13 > sma26 and rsi < RSI_OVERBOUGHT:
        sinyal = "MOM AL"
    else:
        sinyal = "BEKLE"

    # Adım 2: Kalite filtresi — düşük skor veya yetersiz geçmiş → BEKLE
    # Bu filtre "çok fazla sinyal" sorununu çözer:
    # Boğa piyasasında 300 hisse AL alabilir; biz sadece en güçlüleri istiyoruz
    if sinyal in ("DİP FIRSATI", "GÜÇLÜ AL", "AL", "HMM AL", "MOM AL"):
        if composite < MIN_COMPOSITE:
            sinyal = "BEKLE"          # Skor yetmedi
        elif wfo["sharpe"] < MIN_WFO_SHARPE:
            sinyal = "BEKLE"          # Tarihsel kanıt yetersiz

    # ── Beklenen Değer (Expected Value) ─────────────────────────────────────
    # EV: stop_koşulunu baz alarak P(kazanç) × hedef + P(kayıp) × stop_kaybı
    stop_price = cur - 2 * atr
    gain_pct   = max((tgt - cur) / cur * 100, 0.0) if cur > 0 else 0.0
    loss_pct   = max((cur - stop_price) / cur * 100, 0.1) if cur > 0 else 1.0
    ev_pct     = round(prob * gain_pct - (1 - prob) * loss_pct, 1)

    return dict(
        ticker=ticker,
        fiyat=round(cur, 2),
        sinyal=sinyal,
        hmm_prob=round(prob * 100, 1),
        wfo_sharpe=wfo["sharpe"],
        wfo_ret=wfo["ret"],
        mom13w=round(mom13 * 100, 1),
        mom3w=round(mom3 * 100, 1),
        rsi=round(rsi, 1),
        ext_pct=round(ext_pct, 1),
        top_dist=round(top_dist_pct, 1),
        atr_pct=atr_pct,
        mom_slowing=mom_slowing,
        vol_declining=vol_declining,
        vol_ratio=round(vol, 2),
        kal_vel=round(vel_pct, 2),
        kal_hedef=tgt,
        kal_lo=tgt_lo,
        kal_hi=tgt_hi,
        kal_upside=round(upside, 1),
        gain_pct=round(gain_pct, 1),
        loss_pct=round(loss_pct, 1),
        ev_pct=ev_pct,
        weeks_bull=weeks_in_bull,
        pullback=is_pullback,
        fresh=is_fresh,
        stop=round(stop_price, 2),
        composite=composite,
    )

print("✅ scan_ticker hazır.")
print(f"   Kalite filtresi: composite >= {MIN_COMPOSITE} VE wfo_sharpe >= {MIN_WFO_SHARPE}")
print(f"   Beklenen Değer (EV) = P(bull) × hedef_getiri% - P(ayı) × stop_kaybı%")


In [ ]:
print(f"🔍 Tarama başlıyor: {len(WEEKLY)} hisse\n")

results, errors = [], []
for ticker in tqdm(list(WEEKLY.keys()), desc="HMM+WFO Tarama"):
    try:
        r = scan_ticker(ticker)
        if r: results.append(r)
    except Exception as e:
        errors.append((ticker, str(e)))

DF = (pd.DataFrame(results)
        .sort_values("composite", ascending=False)
        .reset_index(drop=True))
DF.index += 1

# Sinyal özeti
print("\n" + "="*60)
SINYAL_ORDER = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL","MOM AL","AŞIRI DEĞER","BEKLE"]
EMOJI = {"DİP FIRSATI":"⭐","GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵",
         "MOM AL":"🟣","AŞIRI DEĞER":"⚠️","BEKLE":"🔴"}
for s in SINYAL_ORDER:
    n = len(DF[DF["sinyal"]==s])
    if n: print(f"  {EMOJI.get(s,'')} {s:<15}: {n:>4} hisse")
print("="*60)


In [ ]:
date_str = pd.Timestamp.today().strftime("%Y-%m-%d")

# ── Sinyal özeti ────────────────────────────────────────────────────────────
print("═" * 75)
print(f"  BIST HMM+MOM3 TARAMA SONUÇLARI — {date_str}")
print(f"  Kalite Filtresi: composite ≥ {MIN_COMPOSITE}  |  WFO Sharpe ≥ {MIN_WFO_SHARPE}")
print("═" * 75)

SINYAL_ORDER = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL","MOM AL","AŞIRI DEĞER","BEKLE"]
EMOJI = {"DİP FIRSATI":"⭐","GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵",
         "MOM AL":"🟣","AŞIRI DEĞER":"⚠️","BEKLE":"⬜"}
for s in SINYAL_ORDER:
    n = len(DF[DF["sinyal"]==s])
    bar = "█" * min(n // 3, 30)
    if n: print(f"  {EMOJI.get(s,'')} {s:<15}: {n:>4} hisse  {bar}")
print("═" * 75)
total_buy = len(DF[DF["sinyal"].isin(["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL","MOM AL"])])
print(f"  Toplam AL sinyali: {total_buy} | Toplam taranan: {len(DF)}")
print()

# ── Görüntülenecek sütunlar ─────────────────────────────────────────────────
DCOLS = ["ticker","fiyat","sinyal","composite","hmm_prob","wfo_sharpe",
         "mom13w","rsi","ext_pct","top_dist","kal_hedef","kal_upside","ev_pct",
         "weeks_bull","stop","vol_ratio"]

def show_section(label, mask, emoji="", max_n=TOP_N_PER_SIG):
    sub = DF[mask].head(max_n)
    if sub.empty: return
    print(f"\n{emoji} {label} ({len(DF[mask])} hisse, en iyi {min(len(DF[mask]), max_n)} gösteriliyor)")
    print("─" * 115)
    print(sub[DCOLS].to_string(index=True))

# ⭐ DİP FIRSATI — en öncelikli
show_section("DİP FIRSATI  — Boğa trendinde pullback: EN İYİ GİRİŞ NOKTASI",
             DF["sinyal"]=="DİP FIRSATI", "⭐")

# 🟢 GÜÇLÜ AL
show_section("GÜÇLÜ AL  — Tüm koşullar ideal, uzamamış",
             DF["sinyal"]=="GÜÇLÜ AL", "🟢")

# 🟡 AL
show_section("AL  — HMM boğa + pozitif momentum, ext<%20",
             DF["sinyal"]=="AL", "🟡")

# 🔵 HMM AL (sadece iyi WFO'lu olanlar)
show_section("HMM AL  — HMM boğa rejimi (diğer koşullar kısmi)",
             (DF["sinyal"]=="HMM AL") & (DF["wfo_sharpe"] >= 0.5), "🔵",
             max_n=10)

# ⚠️ AŞIRI DEĞER
asiri = DF[DF["sinyal"]=="AŞIRI DEĞER"]
if len(asiri):
    print(f"\n⚠️  AŞIRI DEĞER ({len(asiri)} hisse) — Tükenme sinyali, YENİ POZİSYON AÇMAYIN")
    print("─" * 95)
    acols = ["ticker","fiyat","composite","hmm_prob","rsi","ext_pct","top_dist",
             "mom13w","mom_slowing","vol_declining","wfo_sharpe"]
    print(asiri.head(TOP_N_PER_SIG)[acols].to_string(index=True))

print()

# ── Sütun açıklamaları ───────────────────────────────────────────────────────
print("─" * 75)
print("Sütun Açıklamaları:")
print("  composite  = Toplam kalite skoru (0-100) — ne yüksekse o kadar iyi")
print("  hmm_prob   = HMM boğa rejimi olasılığı %")
print("  wfo_sharpe = Walk-Forward geçmiş Sharpe  (>1=güçlü, >0=pozitif)")
print("  mom13w     = 13 haftalık momentum %  (ideal: %5-70)")
print("  rsi        = RSI(14)  — >70 dikkat, <30 aşırı satım")
print("  ext_pct    = Fiyat / SMA26 farkı % (>20 aşırı uzamış)")
print("  top_dist   = 52H zirvesine uzaklık %  (düşük = zirveye yakın)")
print("  kal_hedef  = Kalman 13H fiyat hedefi")
print("  kal_upside = Hedefe kaç % potansiyel")
print("  ev_pct     = Beklenen değer % = P(bull)×hedef - P(ayı)×stop_kaybı")
print("  weeks_bull = Bu boğa rejiminde kaç haftadır (1-4=taze)")
print("  stop       = 2×ATR stop fiyatı")
print("  vol_ratio  = Hacim / 20H ort  (>1.5 yüksek katılım)")

# ── CSV kaydet ───────────────────────────────────────────────────────────────
csv_path = f"bist_tarama_{date_str}.csv"
DF.to_csv(csv_path, index=True)
print(f"\n💾 Tüm sonuçlar CSV'ye kaydedildi: {csv_path}")


In [ ]:
# En iyi giriş noktaları: DİP FIRSATI önce, sonra GÜÇLÜ AL
priority_sigs = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL"]
plot_df = pd.concat([DF[DF["sinyal"]==s] for s in priority_sigs]).head(20)
if len(plot_df) == 0: plot_df = DF.head(20)

n = len(plot_df); ncols = 5
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(24, nrows*4))
axes = axes.flatten()
fig.suptitle(
    f"BIST — Giriş Zamanlamalı HMM+MOM3 En İyi {n} Sinyal | {date_str}\n"
    "⭐=Dip Fırsatı 🟢=Güçlü AL | Sarı bölge=aşırı uzama eşiği | Mavi=Kalman hedef",
    fontsize=12, fontweight="bold"
)
hmm_cache = {}

for idx, (_, row) in enumerate(plot_df.iterrows()):
    ax = axes[idx]; tkr = row["ticker"]
    if tkr not in WEEKLY: ax.set_visible(False); continue
    df = WEEKLY[tkr].tail(104)
    if tkr not in hmm_cache:
        h = HMMStrategy(); h.fit(WEEKLY[tkr]); hmm_cache[tkr] = h
    states = hmm_cache[tkr].predict_states(WEEKLY[tkr]).reindex(df.index).fillna(0)

    ax.plot(df.index, df["close"], color="black", lw=1.3, zorder=5)
    for j in range(len(df)-1):
        c = "#d0f5d0" if states.iloc[j]==1 else "#f5d0d0"
        ax.axvspan(df.index[j], df.index[j+1], alpha=0.3, color=c, zorder=1)
    ax.plot(df.index, df["sma13"], color="darkorange", lw=0.9, alpha=0.85)
    ax.plot(df.index, df["sma26"], color="purple",     lw=0.9, alpha=0.85)

    # Aşırı uzama eşiği (SMA26 × 1.20) — sarı çizgi
    if "sma26" in df.columns:
        ax.plot(df.index, df["sma26"]*1.20, color="gold", lw=0.7,
                ls="--", alpha=0.7, label="+20%SMA")

    ax.axhline(row["kal_hedef"], color="royalblue", ls="--", lw=1.0)
    ax.axhline(row["stop"],      color="crimson",   ls=":",  lw=0.8)

    SIG_COLOR = {"DİP FIRSATI":"gold","GÜÇLÜ AL":"darkgreen",
                 "AL":"darkorange","HMM AL":"steelblue"}.get(row["sinyal"],"gray")
    em = {"DİP FIRSATI":"⭐","GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵"}.get(row["sinyal"],"")
    ax.set_title(
        f"{tkr} {em}{row['sinyal']}\n"
        f"Skor:{row['composite']:.0f} RSI:{row['rsi']:.0f} "
        f"Ext:{row['ext_pct']:+.0f}% Hedef:{row['kal_hedef']:.0f}({row['kal_upside']:+.0f}%)",
        fontsize=7.5, color=SIG_COLOR, fontweight="bold"
    )
    ax.tick_params(labelsize=6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

for i in range(idx+1, len(axes)): axes[i].set_visible(False)
plt.tight_layout()
plt.savefig(f"bist_giris_{date_str}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Grafik kaydedildi: bist_giris_{date_str}.png")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📐 STOP / ÇIKIŞ ŞARTLARI SİMÜLASYONU
#
# Her AL sinyali için 3 senaryo + beklenen değer hesabı
#
# Senaryo A — Stop tetiklenir : Fiyat 2×ATR altına düşer, pozisyon kapanır
# Senaryo B — Hedef ulaşılır  : Fiyat Kalman 13H hedefine erişir, al-satış
# Senaryo C — Zaman çıkışı    : 13 hafta (1 çeyrek) sonra sonucu ne olursa ol gözden geçir
#
# Beklenen Değer (EV) = P(bull) × Hedef_Getiri% + P(ayı) × (-Stop_Kaybı%)
# EV > 0 → pozitif beklenti  |  EV > R:R threshold → güçlü fırsat
#
# "Yükselmiş hisseler dağıtım mı?" sorusuna cevap:
# - WFO Sharpe > 1 olanlar geçmişte bu hissede algoritma çalışmış demek
# - EV > %5 ve R:R > 2 → hem tarihsel hem matematiksel olarak savunulabilir
# - mom_slowing=True veya vol_declining=True ise EV düşük çıkar → filtrele
# ═══════════════════════════════════════════════════════════════════════════

buy_signals = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL"]
sim_df = DF[DF["sinyal"].isin(buy_signals)].copy().head(30)  # en iyi 30

if sim_df.empty:
    print("Simülasyon için AL sinyali bulunamadı.")
else:
    print("═" * 110)
    print(f"  📐 STOP / ÇIKIŞ SİMÜLASYONU — {len(sim_df)} Hisse")
    print(f"  Kural: Stop=2×ATR | Hedef=Kalman 13H | Zaman=13 hafta")
    print("═" * 110)

    HDR = (f"  {'Hisse':<8} {'Sinyal':<13} {'Giriş':>8} {'Stop':>8} "
           f"{'Stop%':>7} {'Hedef':>9} {'Hedef%':>8} {'R:R':>5} "
           f"{'P(bull)':>7} {'EV%':>6} "
           f"{'WFO':>6} {'Mom↓':>5} {'Vol↓':>5} {'Yorum'}")
    print(HDR)
    print("─" * 110)

    sim_rows = []
    for _, r in sim_df.iterrows():
        ticker   = str(r["ticker"])
        sinyal   = str(r["sinyal"])
        fiyat    = float(r["fiyat"])
        stop_p   = float(r["stop"])
        hedef    = float(r["kal_hedef"])
        p_bull   = float(r["hmm_prob"]) / 100
        wfo_s    = float(r["wfo_sharpe"])
        mom_s    = bool(r["mom_slowing"])
        vol_d    = bool(r["vol_declining"])
        ev_pct   = float(r["ev_pct"])
        composite= float(r["composite"])

        stop_pct  = round((stop_p - fiyat) / fiyat * 100, 1)   # negatif
        hedef_pct = round((hedef  - fiyat) / fiyat * 100, 1)   # pozitif
        rr        = round(abs(hedef_pct / stop_pct), 2) if stop_pct != 0 else 0

        # Yorum mantığı
        flags = []
        if ev_pct > 8 and rr >= 2.5 and wfo_s >= 1.0:
            yorum = "⭐ GÜÇLÜ FIRSAT"
        elif ev_pct > 4 and rr >= 1.8:
            yorum = "✅ İYİ FIRSAT"
        elif ev_pct > 0 and rr >= 1.5:
            yorum = "👍 KABUL"
        elif ev_pct <= 0:
            yorum = "❌ NEGATİF EV"
        else:
            yorum = "⚠️ ZAYIF R:R"

        if mom_s:  flags.append("Mom↓")
        if vol_d:  flags.append("Hacim↓")
        if flags:  yorum += f"  [{', '.join(flags)}]"

        line = (f"  {ticker:<8} {sinyal:<13} {fiyat:>8.2f} {stop_p:>8.2f} "
                f"{stop_pct:>+6.1f}% {hedef:>9.2f} {hedef_pct:>+7.1f}% "
                f"{rr:>5.2f} {p_bull*100:>6.0f}% {ev_pct:>+5.1f}%  "
                f"{wfo_s:>5.2f}  {'✓' if mom_s else '-':>4}  {'✓' if vol_d else '-':>4}  {yorum}")
        print(line)

        sim_rows.append({
            "Hisse": ticker, "Sinyal": sinyal,
            "Giriş": fiyat, "Stop": stop_p, "Stop%": stop_pct,
            "Hedef": hedef, "Hedef%": hedef_pct,
            "R:R": rr, "P(bull)%": round(p_bull*100,0),
            "EV%": ev_pct, "WFO": wfo_s, "Skor": composite,
            "Mom↓": mom_s, "Vol↓": vol_d, "Yorum": yorum,
        })

    print("─" * 110)

    sim_tbl = pd.DataFrame(sim_rows)

    # Özet istatistikler
    pos_ev   = (sim_tbl["EV%"] > 0).sum()
    high_rr  = (sim_tbl["R:R"] >= 2).sum()
    strong   = ((sim_tbl["EV%"] > 4) & (sim_tbl["R:R"] >= 2)).sum()
    avg_ev   = sim_tbl["EV%"].mean()

    print(f"\n  Özet:")
    print(f"  Pozitif EV (>0)  : {pos_ev}/{len(sim_tbl)} hisse")
    print(f"  İyi R:R (≥2)     : {high_rr}/{len(sim_tbl)} hisse")
    print(f"  Güçlü fırsat     : {strong}/{len(sim_tbl)} hisse (EV>%4 VE R:R≥2)")
    print(f"  Ortalama EV      : %{avg_ev:.1f}")

    print(f"""
  ─────────────────────────────────────────────────────
  ÇIKIŞ KURALLARI (strateji tutarlılığı için önemli):

  STOP  : Fiyat 2×ATR seviyesinin altına kapanırsa hemen çık
          Neden 2×ATR? Normal gürültüden daha büyük bir hareket
          Beklenen kayıp: Stop% sütununda gösterilmekte

  HEDEF : Fiyat Kalman hedefine ulaşırsa %50 sat, kalanı trailing stop ile tut
          Neden %50? Trend devam edebilir; tam çıkış potansiyel kazancı keser

  ZAMAN : 13 hafta sonra (1 çeyrek) pozisyonu gözden geçir
          HMM rejimi hâlâ boğa mı? → Tutmaya devam
          Rejim değişti mi? → Çık (stop tetiklenmemiş olsa bile)

  "YIKILMAK" RİSKİ:
  Dağıtım riskini minimize etmek için kullan:
    ✓ WFO Sharpe ≥ 1.0  → bu hissede algoritma geçmişte çalışmış
    ✓ mom_slowing=Hayır → momentum hâlâ güçlü
    ✓ vol_declining=Hayır → alıcı katılımı devam ediyor
    ✓ EV > %5           → matematiksel olarak savunulabilir beklenti
    ✓ R:R ≥ 2           → kazanç/kayıp oranı mantıklı
  ─────────────────────────────────────────────────────
""")

    # Simülasyon tablosunu CSV'ye ekle
    date_str2 = pd.Timestamp.today().strftime("%Y-%m-%d")
    sim_tbl.to_csv(f"bist_simulasyon_{date_str2}.csv", index=False)
    print(f"  💾 Simülasyon tablosu kaydedildi: bist_simulasyon_{date_str2}.csv")


In [ ]:
print("\n" + "═"*80)
print("💼 QUARTER-KELLY POZİSYON ÖNERİLERİ")
print("   (Sadece DİP FIRSATI + GÜÇLÜ AL + AL sinyalleri)")
print("═"*80)

buy_df = DF[DF["sinyal"].isin(["DİP FIRSATI","GÜÇLÜ AL","AL"])].copy()

if len(buy_df) == 0:
    print("AL sinyali yok. HMM AL listesini inceleyin.")
    buy_df = DF[DF["sinyal"]=="HMM AL"].head(5)

pos_list = []
for _, r in buy_df.head(15).iterrows():
    p    = r["hmm_prob"]/100
    b    = max(r["kal_upside"]/100, 0.05)
    risk = max((r["fiyat"]-r["stop"])/r["fiyat"], 0.01)
    qk   = min(max((p*b-(1-p)*risk)/b, 0)*0.25, 0.15)
    pos_list.append({
        "Hisse":    r["ticker"],
        "Sinyal":   r["sinyal"],
        "Fiyat":    r["fiyat"],
        "Hedef":    r["kal_hedef"],
        "Stop":     r["stop"],
        "RSI":      r["rsi"],
        "Ext%":     r["ext_pct"],
        "ZirveUzk%":r["top_dist"],
        "R:R":      round(b/risk,2) if risk>0 else 0,
        "Pos%":     round(qk*100,1),
        "Skor":     r["composite"],
    })

if pos_list:
    pos_df = pd.DataFrame(pos_list).sort_values("Pos%", ascending=False)
    pos_df.reset_index(drop=True, inplace=True); pos_df.index+=1
    total = pos_df["Pos%"].sum()
    print(pos_df.to_string())
    print("-"*80)
    print(f"Toplam yatırım : %{total:.1f}  |  Nakit: %{100-total:.1f}")
    print("\n⚠️  Bu çıktı yatırım tavsiyesi değildir.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📊 DETAYLI RENKLİ EXCEL ÇIKTISI — 5 Sayfa
#    Sayfa 1: Tüm Sonuçlar  (tüm hisseler, tüm metrikler, renk kodlu)
#    Sayfa 2: AL Sinyalleri (sadece al sinyalleri, öncelik sırası)
#    Sayfa 3: Pozisyon Önerileri (Quarter-Kelly boyutlama, R:R, not)
#    Sayfa 4: Dikkat Listesi (AŞIRI DEĞER hisseleri)
#    Sayfa 5: Açıklamalar  (renk & metrik legend)
# ═══════════════════════════════════════════════════════════════════════════

try:
    import openpyxl
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])
    import openpyxl

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

date_str   = pd.Timestamp.today().strftime("%Y-%m-%d")
excel_path = f"BIST_HMM_Tarama_{date_str}.xlsx"

# ── Stil yardımcıları ─────────────────────────────────────────────────────
def fill(h):
    return PatternFill("solid", fgColor=h.upper().lstrip("#"))

def fnt(size=9, bold=False, color="000000"):
    return Font(name="Calibri", size=size, bold=bold, color=color.lstrip("#"))

def aln(h="center", wrap=False):
    return Alignment(horizontal=h, vertical="center", wrap_text=wrap)

_side  = Side(border_style="thin", color="BFBFBF")
BORDER = Border(left=_side, right=_side, top=_side, bottom=_side)

# ── Sinyal renk tablosu ───────────────────────────────────────────────────
SIG_STYLE = {
    "DİP FIRSATI": ("1F497D", "FFFFFF"),   # lacivert / beyaz
    "GÜÇLÜ AL":    ("00B050", "FFFFFF"),   # yeşil / beyaz
    "AL":          ("70AD47", "FFFFFF"),   # açık yeşil / beyaz
    "HMM AL":      ("2E75B6", "FFFFFF"),   # mavi / beyaz
    "MOM AL":      ("7030A0", "FFFFFF"),   # mor / beyaz
    "AŞIRI DEĞER": ("C00000", "FFFFFF"),   # koyu kırmızı / beyaz
    "BEKLE":       ("767676", "FFFFFF"),   # gri / beyaz
}

def score_fill(s):
    """0-100 → kırmızı-sarı-yeşil gradient"""
    s = max(0.0, min(100.0, float(s or 0)))
    if s < 50:
        r, g = 255, int(180 * s / 50)
    else:
        r, g = int(255 * (100 - s) / 50), 180
    return fill(f"{r:02X}{g:02X}40")

def write_title(ws, text, n_cols, bg="1F3864", fg="FFFFFF"):
    ws.merge_cells(f"A1:{get_column_letter(n_cols)}1")
    c = ws["A1"]
    c.value, c.fill, c.font = text, fill(bg), fnt(12, True, fg)
    c.alignment = aln()
    ws.row_dimensions[1].height = 26

def write_header(ws, col_defs, row=2, bg="2E75B6", fg="FFFFFF"):
    for ci, (label, _, width) in enumerate(col_defs, 1):
        c = ws.cell(row=row, column=ci)
        c.value, c.fill, c.font = label, fill(bg), fnt(9, True, fg)
        c.alignment = aln(wrap=True)
        c.border = BORDER
        ws.column_dimensions[get_column_letter(ci)].width = width
    ws.row_dimensions[row].height = 32
    ws.freeze_panes = f"A{row + 1}"

def to_py(v):
    """numpy scalar → Python native"""
    if isinstance(v, (bool, int, float, str)): return v
    try: return v.item()
    except: return v

# ── Veri hazırlığı ────────────────────────────────────────────────────────
df_xl = DF.copy().reset_index(drop=True)
df_xl.index = range(1, len(df_xl) + 1)

# Bool → okunabilir string
for col in ("mom_slowing", "vol_declining", "pullback", "fresh"):
    if col in df_xl.columns:
        df_xl[col] = df_xl[col].map({True: "✓ Evet", False: "Hayır",
                                      1: "✓ Evet", 0: "Hayır"}).fillna("Hayır")

# ── Sütun tanımları ──────────────────────────────────────────────────────
# (etiket, alan_adı, genişlik)
MAIN_COLS = [
    ("#",             "#",            4 ),
    ("Hisse",         "ticker",      10 ),
    ("Fiyat ₺",      "fiyat",       10 ),
    ("Sinyal",        "sinyal",      15 ),
    ("Skor\n/100",    "composite",    8 ),
    ("HMM\n%",        "hmm_prob",     8 ),
    ("WFO\nSharpe",   "wfo_sharpe",  10 ),
    ("WFO\nGetiri%",  "wfo_ret",     10 ),
    ("MOM\n13H%",     "mom13w",       9 ),
    ("MOM\n3H%",      "mom3w",        9 ),
    ("RSI",           "rsi",          7 ),
    ("Uzama%\n(SMA26)","ext_pct",    11 ),
    ("Zirve\nUzk%",   "top_dist",    10 ),
    ("ATR%",          "atr_pct",      7 ),
    ("Kal.\nHedef",   "kal_hedef",   11 ),
    ("Kal.\nAlt",     "kal_lo",      11 ),
    ("Kal.\nÜst",     "kal_hi",      11 ),
    ("Kal.\nGetiri%", "kal_upside",  10 ),
    ("Kal.\nHız/Y%",  "kal_vel",     10 ),
    ("Boğa\nHafta",   "weeks_bull",   9 ),
    ("Stop ₺",        "stop",        11 ),
    ("Hacim\nOran",   "vol_ratio",    9 ),
    ("Mom\nYavaş",    "mom_slowing", 10 ),
    ("Dağıtım",      "vol_declining", 9 ),
    ("Pullback",      "pullback",     9 ),
    ("Taze\nSinyal",  "fresh",        9 ),
]

# ── Ortak satır yazıcı ────────────────────────────────────────────────────
def write_row(ws, excel_row, series, col_defs, row_bg="FFFFFF"):
    sinyal = str(series.get("sinyal", "BEKLE"))
    for ci, (label, field, _) in enumerate(col_defs, 1):
        c = ws.cell(row=excel_row, column=ci)
        val = series.get(field, "") if field != "#" else series.get("#", excel_row - 2)
        c.value    = to_py(val)
        c.border   = BORDER
        c.font     = fnt(9)
        c.fill     = fill(row_bg)
        c.alignment = aln()

        if field == "sinyal":
            bg, fg_ = SIG_STYLE.get(sinyal, ("CCCCCC", "000000"))
            c.fill = fill(bg); c.font = fnt(9, True, fg_)
        elif field == "composite":
            c.fill = score_fill(val); c.font = fnt(9, True); c.number_format = "0.0"
        elif field in ("fiyat", "kal_hedef", "kal_lo", "kal_hi", "stop"):
            c.number_format = "#,##0.00"
        elif field == "rsi" and isinstance(c.value, float):
            c.number_format = "0.0"
            v = c.value
            if v > 70:   c.fill = fill("FF9900"); c.font = fnt(9, True)
            elif v > 60: c.fill = fill("FFFACD")
            elif v < 30: c.fill = fill("BDD7EE")
        elif field == "ext_pct" and isinstance(c.value, float):
            c.number_format = "0.0"
            v = c.value
            if v > 35:   c.fill = fill("FF0000"); c.font = fnt(9, True, "FFFFFF")
            elif v > 20: c.fill = fill("FF9900"); c.font = fnt(9, True)
            elif v > 10: c.fill = fill("FFFF00")
        elif field == "kal_upside" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value > 20:  c.fill = fill("00B050"); c.font = fnt(9, color="FFFFFF")
            elif c.value < 0: c.fill = fill("FF0000"); c.font = fnt(9, color="FFFFFF")
        elif field == "wfo_sharpe" and isinstance(c.value, float):
            c.number_format = "0.000"
            if c.value > 1.0:  c.fill = fill("00B050"); c.font = fnt(9, color="FFFFFF")
            elif c.value > 0.5:c.fill = fill("E2EFDA")
            elif c.value < 0:  c.fill = fill("FFE0CC")
        elif field == "vol_ratio" and isinstance(c.value, float):
            c.number_format = "0.00"
            if c.value > 2.0:  c.fill = fill("00B050"); c.font = fnt(9, color="FFFFFF")
            elif c.value < 0.75: c.fill = fill("FF9900")
        elif field in ("mom_slowing", "vol_declining") and isinstance(c.value, str):
            if "Evet" in c.value:
                c.fill = fill("FF9900"); c.font = fnt(9, True)
        elif field in ("pullback", "fresh") and isinstance(c.value, str):
            if "Evet" in c.value:
                c.fill = fill("00B050"); c.font = fnt(9, True, "FFFFFF")
        elif field in ("hmm_prob","mom13w","mom3w","top_dist","atr_pct","wfo_ret","kal_vel"):
            c.number_format = "0.0"
        elif field == "weeks_bull":
            c.number_format = "0"
            if isinstance(c.value, int) and 0 < c.value <= 4:
                c.fill = fill("E2EFDA")  # taze sinyal hafif yeşil

# ══════════════════════════════════════════════════════════════════════════
# SAYFA 1 — TÜM SONUÇLAR
# ══════════════════════════════════════════════════════════════════════════
wb = Workbook()
ws1 = wb.active
ws1.title = "Tüm Sonuçlar"
n = len(MAIN_COLS)
write_title(ws1, f"🔍 BIST HMM+MOM3 TAM TARAMA — {date_str}  |  Toplam: {len(df_xl)} Hisse", n)
write_header(ws1, MAIN_COLS)

for i, (ridx, row) in enumerate(df_xl.iterrows(), 1):
    bg = "F5F5F5" if i % 2 == 0 else "FFFFFF"
    row_dict = row.to_dict()
    row_dict["#"] = i
    write_row(ws1, i + 2, row_dict, MAIN_COLS, bg)

ws1.auto_filter.ref = f"A2:{get_column_letter(n)}2"

# ══════════════════════════════════════════════════════════════════════════
# SAYFA 2 — AL SİNYALLERİ
# ══════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("AL Sinyalleri")
buy_sigs = ["DİP FIRSATI", "GÜÇLÜ AL", "AL", "HMM AL", "MOM AL"]
df_buy = df_xl[df_xl["sinyal"].isin(buy_sigs)].copy()

write_title(ws2, f"✅ AL SİNYALLERİ — {date_str}  |  {len(df_buy)} Hisse  |  Skor'a Göre Sıralı", n)
write_header(ws2, MAIN_COLS)

for i, (_, row) in enumerate(df_buy.iterrows(), 1):
    bg = "F0FAF0" if i % 2 == 0 else "FFFFFF"
    row_dict = row.to_dict()
    row_dict["#"] = i
    write_row(ws2, i + 2, row_dict, MAIN_COLS, bg)

ws2.auto_filter.ref = f"A2:{get_column_letter(n)}2"

# ══════════════════════════════════════════════════════════════════════════
# SAYFA 3 — POZİSYON ÖNERİLERİ
# ══════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("Pozisyon Önerileri")

POS_COLS = [
    ("#",              "#",          4 ),
    ("Hisse",          "ticker",    10 ),
    ("Sinyal",         "sinyal",    15 ),
    ("Fiyat ₺",       "fiyat",     10 ),
    ("Kal. Hedef",     "kal_hedef", 12 ),
    ("Stop ₺",         "stop",      10 ),
    ("Upside%",        "upside_p",  10 ),
    ("Downside%",      "down_p",    10 ),
    ("R : R",          "rr",         8 ),
    ("Poz. %",         "pos_pct",   10 ),
    ("HMM %",          "hmm_prob",   9 ),
    ("RSI",            "rsi",        7 ),
    ("Uzama%",         "ext_pct",   10 ),
    ("ATR%",           "atr_pct",    7 ),
    ("WFO Sharpe",     "wfo_sharpe", 10),
    ("Skor",           "composite",  8 ),
    ("Boğa\nHafta",    "weeks_bull",  9),
    ("Not",            "note",       30),
]

pos_rows = []
for _, r in df_xl[df_xl["sinyal"].isin(["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL"])].iterrows():
    fiyat_v  = float(to_py(r.get("fiyat", 1)) or 1)
    stop_v   = float(to_py(r.get("stop", fiyat_v * 0.95)) or fiyat_v * 0.95)
    tgt_v    = float(to_py(r.get("kal_hedef", fiyat_v)) or fiyat_v)
    hmm_v    = float(to_py(r.get("hmm_prob", 60)) or 60) / 100
    upside_v = max((tgt_v / fiyat_v - 1) if fiyat_v > 0 else 0.10, 0.05)
    risk_v   = max((fiyat_v - stop_v) / fiyat_v if fiyat_v > 0 else 0.05, 0.01)
    down_v   = (stop_v / fiyat_v - 1) * 100 if fiyat_v > 0 else -5
    rr       = round(upside_v / risk_v, 2) if risk_v > 0 else 0
    qk       = min(max((hmm_v * upside_v - (1 - hmm_v) * risk_v) / upside_v, 0) * 0.25, 0.15)

    note_parts = []
    if "Evet" in str(r.get("pullback","")):    note_parts.append("Pullback ✓")
    if "Evet" in str(r.get("fresh","")):        note_parts.append("Taze ✓")
    if "Evet" in str(r.get("mom_slowing","")): note_parts.append("⚠ MomYavaş")
    if "Evet" in str(r.get("vol_declining","")): note_parts.append("⚠ HacimDüş")

    pos_rows.append({
        "ticker":    to_py(r["ticker"]),
        "sinyal":    to_py(r["sinyal"]),
        "fiyat":     fiyat_v,
        "kal_hedef": tgt_v,
        "stop":      stop_v,
        "upside_p":  round(upside_v * 100, 1),
        "down_p":    round(down_v, 1),
        "rr":        rr,
        "pos_pct":   round(qk * 100, 1),
        "hmm_prob":  round(hmm_v * 100, 1),
        "rsi":       to_py(r.get("rsi", 0)),
        "ext_pct":   to_py(r.get("ext_pct", 0)),
        "atr_pct":   to_py(r.get("atr_pct", 0)),
        "wfo_sharpe":to_py(r.get("wfo_sharpe", 0)),
        "composite": to_py(r.get("composite", 0)),
        "weeks_bull":to_py(r.get("weeks_bull", 0)),
        "note":      " | ".join(note_parts),
    })

pos_df3 = (pd.DataFrame(pos_rows)
             .sort_values("pos_pct", ascending=False)
             .reset_index(drop=True))
pos_df3.index = range(1, len(pos_df3) + 1)
total_pos = round(pos_df3["pos_pct"].sum(), 1)

np3 = len(POS_COLS)
write_title(ws3,
    f"💼 QUARTER-KELLY POZİSYON ÖNERİLERİ — {date_str}  |  "
    f"Toplam Yatırım: %{total_pos}  |  Nakit: %{round(100 - total_pos, 1)}",
    np3, bg="1F497D")
write_header(ws3, POS_COLS, bg="1F497D")

for i, (_, row) in enumerate(pos_df3.iterrows(), 1):
    bg = "EBF3E8" if i % 2 == 0 else "FFFFFF"
    sinyal = str(row.get("sinyal", ""))
    for ci, (label, field, _) in enumerate(POS_COLS, 1):
        c = ws3.cell(row=i + 2, column=ci)
        val = row.get(field, "") if field != "#" else i
        c.value = to_py(val)
        c.border = BORDER
        c.font   = fnt(9)
        c.fill   = fill(bg)
        c.alignment = aln()

        if field == "sinyal":
            bg2, fg2 = SIG_STYLE.get(sinyal, ("CCCCCC", "000000"))
            c.fill = fill(bg2); c.font = fnt(9, True, fg2)
        elif field in ("fiyat", "kal_hedef", "stop"):
            c.number_format = "#,##0.00"
        elif field == "composite":
            c.fill = score_fill(c.value); c.number_format = "0.0"
        elif field == "pos_pct" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value >= 12: c.fill = fill("00B050"); c.font = fnt(9, True, "FFFFFF")
            elif c.value >= 7: c.fill = fill("92D050"); c.font = fnt(9, True)
            elif c.value >= 3: c.fill = fill("E2EFDA")
        elif field == "rr" and isinstance(c.value, float):
            c.number_format = "0.00"
            if c.value >= 3:   c.fill = fill("00B050"); c.font = fnt(9, True, "FFFFFF")
            elif c.value >= 2: c.fill = fill("E2EFDA")
            elif c.value < 1:  c.fill = fill("FFE0CC")
        elif field == "upside_p" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value > 20: c.fill = fill("92D050")
        elif field == "down_p" and isinstance(c.value, float):
            c.number_format = "0.0"
            c.fill = fill("FFE0CC")
        elif field == "rsi" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value > 70: c.fill = fill("FF9900")
        elif field in ("hmm_prob","ext_pct","atr_pct","wfo_sharpe"):
            c.number_format = "0.0" if field != "wfo_sharpe" else "0.000"
        elif field == "note":
            c.alignment = aln("left")
            if "⚠" in str(c.value): c.font = fnt(9, color="C00000")

# Toplam / Nakit satırları
tr = len(pos_df3) + 4
ws3.cell(tr, 1, "TOPLAM").font = fnt(10, True)
c_tot = ws3.cell(tr, 10, total_pos)
c_tot.font = fnt(10, True); c_tot.fill = fill("FFD700"); c_tot.number_format = "0.0"
c_tot.border = BORDER

ws3.cell(tr + 1, 1, "NAKİT").font = fnt(10, True)
c_cas = ws3.cell(tr + 1, 10, round(100 - total_pos, 1))
c_cas.font = fnt(10, True); c_cas.fill = fill("BDD7EE"); c_cas.number_format = "0.0"
c_cas.border = BORDER

# ══════════════════════════════════════════════════════════════════════════
# SAYFA 4 — DİKKAT LİSTESİ / AŞIRI DEĞER
# ══════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet("Dikkat Listesi")
df_warn = df_xl[df_xl["sinyal"] == "AŞIRI DEĞER"].copy()

WARN_COLS = [
    ("#",             "#",            4 ),
    ("Hisse",         "ticker",      10 ),
    ("Fiyat ₺",      "fiyat",       10 ),
    ("Sinyal",        "sinyal",      15 ),
    ("RSI",           "rsi",          7 ),
    ("Uzama%\n(SMA26)","ext_pct",    11 ),
    ("Zirve\nUzk%",   "top_dist",    10 ),
    ("MOM 13H%",      "mom13w",      10 ),
    ("MOM  3H%",      "mom3w",        9 ),
    ("Mom\nYavaş",    "mom_slowing", 10 ),
    ("Dağıtım\nPaterni","vol_declining",11),
    ("Hacim\nOran",   "vol_ratio",    9 ),
    ("HMM %",         "hmm_prob",     8 ),
    ("Kal. Hedef",    "kal_hedef",   11 ),
    ("Stop ₺",        "stop",        11 ),
    ("Skor",          "composite",    8 ),
    ("Uyarı Notu",    "warn_note",   35 ),
]

def build_warn_note(row):
    parts = []
    if "Evet" in str(row.get("mom_slowing","")): parts.append("Momentum yavaşlıyor")
    if "Evet" in str(row.get("vol_declining","")): parts.append("Hacim azalıyor")
    rsi_v = to_py(row.get("rsi", 0))
    if isinstance(rsi_v, float) and rsi_v > 70: parts.append(f"RSI {rsi_v:.0f} > 70")
    ext_v = to_py(row.get("ext_pct", 0))
    if isinstance(ext_v, float) and ext_v > 20: parts.append(f"Ext +{ext_v:.0f}%")
    top_v = to_py(row.get("top_dist", 99))
    if isinstance(top_v, float) and top_v < 5:  parts.append(f"Zirveye %{top_v:.1f}")
    return " | ".join(parts) if parts else "Çoklu tükenme sinyali"

df_warn = df_warn.copy()
df_warn["warn_note"] = df_warn.apply(build_warn_note, axis=1)

nw = len(WARN_COLS)
write_title(ws4,
    f"⚠️  DİKKAT — AŞIRI DEĞER / TÜKENME SİNYALİ — {date_str}  |  "
    f"{len(df_warn)} Hisse  |  YENİ POZİSYON AÇMAYIN",
    nw, bg="C00000")
write_header(ws4, WARN_COLS, bg="C00000")

for i, (_, row) in enumerate(df_warn.iterrows(), 1):
    bg = "FFF0F0" if i % 2 == 0 else "FFFFFF"
    row_dict = row.to_dict()
    row_dict["#"] = i
    for ci, (label, field, _) in enumerate(WARN_COLS, 1):
        c = ws4.cell(row=i + 2, column=ci)
        val = row_dict.get(field, "") if field != "#" else i
        c.value = to_py(val)
        c.border = BORDER
        c.font   = fnt(9)
        c.fill   = fill(bg)
        c.alignment = aln()

        if field == "sinyal":
            c.fill = fill("C00000"); c.font = fnt(9, True, "FFFFFF")
        elif field == "rsi" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value > 70: c.fill = fill("FF0000"); c.font = fnt(9, True, "FFFFFF")
            elif c.value > 60: c.fill = fill("FF9900")
        elif field == "ext_pct" and isinstance(c.value, float):
            c.number_format = "0.0"
            if c.value > 35: c.fill = fill("FF0000"); c.font = fnt(9, True, "FFFFFF")
            elif c.value > 20: c.fill = fill("FF9900"); c.font = fnt(9, True)
        elif field in ("mom_slowing","vol_declining") and isinstance(c.value, str):
            if "Evet" in c.value: c.fill = fill("FF0000"); c.font = fnt(9, True, "FFFFFF")
        elif field in ("fiyat","kal_hedef","stop"):
            c.number_format = "#,##0.00"
        elif field == "composite":
            c.fill = score_fill(c.value); c.number_format = "0.0"
        elif field in ("mom13w","mom3w","top_dist","hmm_prob","vol_ratio"):
            c.number_format = "0.0" if field != "vol_ratio" else "0.00"
        elif field == "warn_note":
            c.alignment = aln("left")
            c.font = fnt(9, color="C00000")

# ══════════════════════════════════════════════════════════════════════════
# SAYFA 5 — AÇIKLAMALAR / LEGEND
# ══════════════════════════════════════════════════════════════════════════
ws5 = wb.create_sheet("Açıklamalar")
ws5.column_dimensions["A"].width = 22
ws5.column_dimensions["B"].width = 62

LEGEND = [
    # (sütun A, sütun B, arka plan, yazı rengi, bold)
    (f"BIST HMM Tarama",       f"Tarih: {date_str}  |  Toplam Taranan: {len(df_xl)} hisse", "1F3864","FFFFFF", True),
    ("","","","",False),
    ("── SİNYALLER ──","","2E75B6","FFFFFF",True),
    ("DİP FIRSATI ⭐",   "HMM boğa rejimi + mom3w<-2% + RSI<70 → Pullback = EN İYİ GİRİŞ",      "1F497D","FFFFFF",True),
    ("GÜÇLÜ AL 🟢",      "HMM boğa + MOM %5-70 + RSI<70 + Ext<%20 + Altın çapraz",               "00B050","FFFFFF",True),
    ("AL 🟡",             "HMM boğa + MOM>0 + RSI<70 + Ext<%20",                                  "70AD47","FFFFFF",True),
    ("HMM AL 🔵",        "HMM boğa rejimi aktif + RSI<70 (diğer koşullar kısmi)",                 "2E75B6","FFFFFF",True),
    ("MOM AL",           "Altın çapraz + MOM>%5 + RSI<70 — HMM bull sinyali yok",                 "7030A0","FFFFFF",True),
    ("AŞIRI DEĞER ⚠️",  "RSI>70 + (Mom yavaş VEYA Hacim düşük) — YENİ POZİSYON AÇMAYIN",        "C00000","FFFFFF",True),
    ("BEKLE",            "Koşullar henüz oluşmamış — Hisseyi takipte tutun",                       "767676","FFFFFF",True),
    ("","","","",False),
    ("── METRİKLER ──","","2E75B6","FFFFFF",True),
    ("Skor /100",        "0=kırmızı → 50=sarı → 100=yeşil | HMM(40) + MOM(20) + Kalman(15) + Giriş(15) + MA(5) + Vol(5) + Bonus(8)","FFFFFF","000000",False),
    ("Uzama% (SMA26)",   ">%10 sarı, >%20 turuncu, >%35 kırmızı → fiyat ne kadar SMA26'nın üzerinde","FFFFFF","000000",False),
    ("RSI",              ">70 turuncu (tek başına aşırı değer DEĞİL), <30 mavi (aşırı satım)","FFFFFF","000000",False),
    ("Kal. Getiri%",     ">%20 yeşil, <0 kırmızı → Kalman filtresi ile 13 haftalık fiyat tahmini","FFFFFF","000000",False),
    ("WFO Sharpe",       ">1.0 yeşil, >0.5 açık yeşil → Walk-Forward Optimization geçmiş performansı","FFFFFF","000000",False),
    ("Hacim Oranı",      ">2 yeşil (güçlü katılım), <0.75 turuncu (dağıtım / alıcı çekilmesi)","FFFFFF","000000",False),
    ("Poz. %",           ">%12 koyu yeşil, >%7 yeşil → Quarter-Kelly formülü ile hesaplanan pozisyon büyüklüğü","FFFFFF","000000",False),
    ("R : R",            ">3 yeşil, >2 açık yeşil, <1 turuncu → Ödül/Risk oranı (hedef / stop)","FFFFFF","000000",False),
    ("ATR%",             "ATR/Fiyat% → Stop büyüklüğü referansı; düşük = sıkı stop mümkün","FFFFFF","000000",False),
    ("Boğa Hafta",       "1-4 arası açık yeşil → taze sinyal; yüksek değer = köklü trend","FFFFFF","000000",False),
    ("","","","",False),
    ("── UYARI ──",      "Bu çıktı algoritmik sinyal aracıdır. Yatırım tavsiyesi değildir.", "FF0000","FFFFFF",True),
    ("Sorumluluk Red",   "Geçmiş performans gelecek getiriyi garanti etmez. Kayıp riski her zaman mevcuttur.", "FF0000","FFFFFF",False),
]

for ri, (a, b, bg_, fg_, bold_) in enumerate(LEGEND, 1):
    ca = ws5.cell(ri, 1, a)
    cb = ws5.cell(ri, 2, b)
    if bg_:
        for c_ in (ca, cb):
            c_.fill = fill(bg_)
            c_.font = fnt(9, bold_, fg_)
            c_.border = BORDER
            c_.alignment = aln("left")
    ws5.row_dimensions[ri].height = 18

# ── Kaydet ────────────────────────────────────────────────────────────────
wb.save(excel_path)

print(f"✅ Excel kaydedildi: {excel_path}")
print()
print("📋 Sayfa İçerikleri:")
buy_n  = len(df_xl[df_xl["sinyal"].isin(buy_sigs)])
warn_n = len(df_xl[df_xl["sinyal"] == "AŞIRI DEĞER"])
print(f"   1. Tüm Sonuçlar       — {len(df_xl):>4} hisse | tüm metrikler, filtre açık")
print(f"   2. AL Sinyalleri      — {buy_n:>4} hisse | DİP FIRSATI > GÜÇLÜ AL > AL > HMM AL > MOM AL")
print(f"   3. Pozisyon Önerileri — {len(pos_df3):>4} hisse | R:R, Poz%, Nakit kalan")
print(f"   4. Dikkat Listesi     — {warn_n:>4} hisse | AŞIRI DEĞER uyarısı")
print(f"   5. Açıklamalar        — Renk kodu & metrik legend")
print()
print("🎨 Renk Kodları:")
for sig, (bg, fg) in SIG_STYLE.items():
    print(f"   #{bg} = {sig}")
print()
print("📊 Skor Renkleri: 0-49 = kırmızı → turuncu, 50-74 = sarı, 75-100 = yeşil")
